### Self-Attention: The Big Picture

Think of self-attention as an **API**.

> **What does it do?**
>
> A token's representation **evolves** by selectively incorporating information from itself and other tokens in the available context.
>
> **Input:** One vector per token, representing what we know about it so far.
>
> **Output:** One updated vector per token, incorporating information from the available context.

**Note:**

In the first layer, the input consists of token embeddings and positional information. In later layers, the input is already context-aware.


In [7]:
import torch
import math

torch.set_printoptions(precision=3, sci_mode=False)

# 3 tokens, each token has a 4D embedding
X = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 2.0],
    [1.0, 1.0, 0.0, 0.0],
])

# Project 4D embeddings into 2D Q, K, V vectors
W_Q = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [0.0, 1.0],
])

W_K = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.0, 1.0],
    [1.0, 0.0],
])

W_V = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.0, 1.0],
])

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

scores = Q @ K.T
scaled_scores = scores / math.sqrt(K.shape[-1])

attention = torch.softmax(scaled_scores, dim=-1)

output = attention @ V

print("Q:\n", Q)
print("\nK:\n", K)
print("\nV:\n", V)

print("\nRaw scores QK^T:\n", scores)
print("\nScaled scores:\n", scaled_scores)

print("\nAttention weights:\n", attention)
print("\nOutput:\n", output)

Q:
 tensor([[2., 0.],
        [0., 4.],
        [1., 1.]])

K:
 tensor([[1., 1.],
        [2., 2.],
        [1., 1.]])

V:
 tensor([[2., 1.],
        [0., 4.],
        [1., 1.]])

Raw scores QK^T:
 tensor([[2., 4., 2.],
        [4., 8., 4.],
        [2., 4., 2.]])

Scaled scores:
 tensor([[1.414, 2.828, 1.414],
        [2.828, 5.657, 2.828],
        [1.414, 2.828, 1.414]])

Attention weights:
 tensor([[0.164, 0.673, 0.164],
        [0.053, 0.894, 0.053],
        [0.164, 0.673, 0.164]])

Output:
 tensor([[0.491, 3.019],
        [0.159, 3.683],
        [0.491, 3.019]])


## Why need to divide sqrt(d_k)?

assume 1 dot product variance is 1

d_k of them variance is dk, std is sqrt(d_k).

> Next step is softmax

Remember scale up and down all softmax input will influence score?

we don't want that, so reverse the variance/std scale up, make std back to 1.

In [23]:

import torch
import math

torch.manual_seed(0)

print(f"{'d':>6} | {'Std':>10} | {'Scaled Std':>12}")
print("-" * 36)

for d in [4, 16, 64, 256]:
    q = torch.randn(100_000, d)
    k = torch.randn(100_000, d)

    scores = (q * k).sum(dim=-1)
    scaled_scores = scores / math.sqrt(d)

    print(f"{d:>6} | {scores.std().item():>10.4f} | {scaled_scores.std().item():>12.4f}")


     d |        Std |   Scaled Std
------------------------------------
     4 |     1.9986 |       0.9993
    16 |     3.9894 |       0.9974
    64 |     8.0005 |       1.0001
   256 |    16.0256 |       1.0016
